# LangGraph Agents with LangSmith
## Agent vs Agentic AI, Telecom Industry Agents, and LangSmith Tracing

This notebook covers:
1. Conceptual difference between Agent and Agentic AI
2. Building agents with LangGraph (Graph API)
3. Real-world telecom industry agent examples
4. Orchestrator pattern with multiple specialized agents
5. LangSmith tracing and observability for agents

All code uses the latest LangGraph and LangChain v0.3+ APIs.

---
## Section 0: Installation and Setup

In [ ]:
# Install all required packages

import subprocess
import sys

packages = [
    "langchain",
    "langchain-openai",
    "langchain-community",
    "langchain-core",
    "langgraph",
    "langsmith",
    "python-dotenv",
]

print("Installing required packages...")
for pkg in packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-U", pkg],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    print(f"  Installed: {pkg}")

print("\nAll packages installed successfully.")

In [ ]:
# Environment variable configuration
# Replace placeholder values with your actual API keys

import os
from dotenv import load_dotenv

load_dotenv(override=False)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "sk-...")

# LangSmith configuration
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "lsv2_...")
os.environ["LANGSMITH_PROJECT"] = "langgraph-agents-telecom"

api_key = os.environ.get("OPENAI_API_KEY", "")
ls_key = os.environ.get("LANGSMITH_API_KEY", "")

if api_key and not api_key.startswith("sk-..."):
    print(f"OpenAI API key loaded: sk-...{api_key[-4:]}")
else:
    print("WARNING: OpenAI API key not set.")

if ls_key and not ls_key.startswith("lsv2_..."):
    print(f"LangSmith API key loaded: lsv2_...{ls_key[-4:]}")
    print(f"LangSmith project: {os.environ['LANGSMITH_PROJECT']}")
    print("LangSmith tracing is ENABLED.")
else:
    print("WARNING: LangSmith API key not set.")

---
## Section 1: Agent vs Agentic AI - Conceptual Overview

### What is an Agent?

An Agent is a software component that uses an LLM to decide which actions (tool calls)
to take in order to accomplish a specific task. It follows a loop:
1. Receive input
2. LLM decides: call a tool or respond
3. If tool call: execute tool, feed result back to LLM
4. Repeat until LLM produces a final response

An agent is task-specific, operates within defined boundaries, and uses a fixed set of tools.

### What is Agentic AI?

Agentic AI is a broader design paradigm where AI systems exhibit autonomous, goal-directed
behavior across complex workflows. Key characteristics:
- **Autonomy**: Makes decisions without step-by-step human guidance
- **Planning**: Breaks complex goals into sub-tasks
- **Tool Use**: Interacts with external systems and APIs
- **Memory**: Maintains state across interactions
- **Collaboration**: Multiple agents work together (orchestration)
- **Adaptation**: Adjusts strategy based on intermediate results

### Key Differences

| Aspect | Agent | Agentic AI |
|--------|-------|------------|
| Scope | Single task, fixed tools | Multi-step workflows, dynamic tool selection |
| Architecture | One LLM + tool loop | Multiple agents, orchestrators, sub-agents |
| Memory | Conversation history | Short-term + long-term persistent memory |
| Planning | Implicit (LLM decides next step) | Explicit (planning, decomposition, delegation) |
| Example | "Check weather in Delhi" | "Monitor network, diagnose faults, escalate tickets" |

In this notebook, we build progressively: simple agents first, then combine them
into agentic systems for telecom use cases.

---
## Section 2: Simple LangGraph Agent - Network Status Checker

A basic telecom agent that can check network status, look up customer plans,
and calculate bills. This demonstrates the core agent loop in LangGraph.

In [ ]:
# ---------------------------------------------------------------
# Tool Definitions for Telecom Agent
# These simulate real telecom operations. In production, these
# would call actual APIs, databases, or monitoring systems.
# ---------------------------------------------------------------

import json
import random
from langchain_core.tools import tool


@tool
def check_network_status(region: str) -> str:
    """Check the current network status for a given region.
    Returns network health metrics including uptime, latency, and active issues.
    
    Args:
        region: The network region to check (e.g., 'North', 'South', 'East', 'West')
    """
    # Simulated network data
    statuses = {
        "north": {"uptime": "99.7%", "latency_ms": 12, "active_issues": 0, "status": "Healthy"},
        "south": {"uptime": "98.2%", "latency_ms": 45, "active_issues": 2, "status": "Degraded"},
        "east": {"uptime": "99.9%", "latency_ms": 8, "active_issues": 0, "status": "Healthy"},
        "west": {"uptime": "95.1%", "latency_ms": 120, "active_issues": 5, "status": "Critical"},
    }
    region_lower = region.lower().strip()
    if region_lower in statuses:
        data = statuses[region_lower]
        return json.dumps({
            "region": region,
            "uptime": data["uptime"],
            "latency_ms": data["latency_ms"],
            "active_issues": data["active_issues"],
            "status": data["status"],
        })
    return json.dumps({"error": f"Unknown region: {region}. Valid regions: North, South, East, West"})


@tool
def lookup_customer(customer_id: str) -> str:
    """Look up customer details by their customer ID.
    Returns plan information, data usage, and account status.
    
    Args:
        customer_id: The unique customer identifier (e.g., 'CUST001')
    """
    customers = {
        "CUST001": {
            "name": "Amit Sharma",
            "plan": "Unlimited 5G",
            "monthly_charge": 999,
            "data_used_gb": 45.2,
            "data_limit_gb": "Unlimited",
            "status": "Active",
        },
        "CUST002": {
            "name": "Priya Patel",
            "plan": "Basic 4G",
            "monthly_charge": 299,
            "data_used_gb": 8.7,
            "data_limit_gb": 10,
            "status": "Active",
        },
        "CUST003": {
            "name": "Rahul Verma",
            "plan": "Enterprise Fiber",
            "monthly_charge": 4999,
            "data_used_gb": 320.5,
            "data_limit_gb": 500,
            "status": "Active",
        },
    }
    cid = customer_id.upper().strip()
    if cid in customers:
        return json.dumps({"customer_id": cid, **customers[cid]})
    return json.dumps({"error": f"Customer not found: {customer_id}"})


@tool
def calculate_bill(customer_id: str, extra_charges: float = 0.0) -> str:
    """Calculate the monthly bill for a customer including any extra charges.
    
    Args:
        customer_id: The unique customer identifier
        extra_charges: Any additional charges to add (default: 0.0)
    """
    base_charges = {
        "CUST001": 999,
        "CUST002": 299,
        "CUST003": 4999,
    }
    cid = customer_id.upper().strip()
    if cid in base_charges:
        base = base_charges[cid]
        tax = round(base * 0.18, 2)  # 18% GST
        total = round(base + tax + extra_charges, 2)
        return json.dumps({
            "customer_id": cid,
            "base_charge": base,
            "tax_gst_18_percent": tax,
            "extra_charges": extra_charges,
            "total_bill": total,
        })
    return json.dumps({"error": f"Customer not found: {customer_id}"})


tools = [check_network_status, lookup_customer, calculate_bill]
print("Telecom tools defined: check_network_status, lookup_customer, calculate_bill")

In [ ]:
# ---------------------------------------------------------------
# Build the LangGraph Agent using the Graph API
# This creates a stateful agent with a tool-calling loop.
# ---------------------------------------------------------------

from typing import Annotated, Literal
from typing_extensions import TypedDict
import operator

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, ToolMessage
from langgraph.graph import StateGraph, START, END


# Define the agent state
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]


# Initialize the LLM with tool binding
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

# Map tool names to tool objects for execution
tools_by_name = {t.name: t for t in tools}


# Node: call the LLM
def call_model(state: AgentState):
    """Invoke the LLM with the current conversation history."""
    system_msg = SystemMessage(
        content=(
            "You are a telecom network operations assistant. "
            "You help operators check network status, look up customer information, "
            "and calculate bills. Be precise and professional in your responses. "
            "Always use the available tools to get accurate data before answering."
        )
    )
    response = llm_with_tools.invoke([system_msg] + state["messages"])
    return {"messages": [response]}


# Node: execute tool calls
def call_tools(state: AgentState):
    """Execute any tool calls made by the LLM."""
    last_message = state["messages"][-1]
    results = []
    for tool_call in last_message.tool_calls:
        tool_fn = tools_by_name[tool_call["name"]]
        result = tool_fn.invoke(tool_call["args"])
        results.append(
            ToolMessage(content=str(result), tool_call_id=tool_call["id"])
        )
    return {"messages": results}


# Conditional edge: decide whether to call tools or finish
def should_continue(state: AgentState) -> Literal["call_tools", "__end__"]:
    """Route to tool execution if the LLM made tool calls, otherwise end."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "call_tools"
    return END


# Build the graph
graph_builder = StateGraph(AgentState)
graph_builder.add_node("call_model", call_model)
graph_builder.add_node("call_tools", call_tools)
graph_builder.add_edge(START, "call_model")
graph_builder.add_conditional_edges("call_model", should_continue, ["call_tools", END])
graph_builder.add_edge("call_tools", "call_model")

# Compile the agent
telecom_agent = graph_builder.compile()

print("Telecom agent compiled successfully.")
print("Graph nodes: call_model -> (conditional) -> call_tools -> call_model -> ... -> END")

In [ ]:
# ---------------------------------------------------------------
# Visualize the Agent Graph (works in VS Code and Jupyter)
# ---------------------------------------------------------------

try:
    from IPython.display import Image, display
    graph_image = telecom_agent.get_graph(xray=True).draw_mermaid_png()
    display(Image(graph_image))
except Exception as e:
    # Fallback: print the mermaid diagram as text
    print("Graph visualization (Mermaid format):")
    print(telecom_agent.get_graph(xray=True).draw_mermaid())

In [ ]:
# ---------------------------------------------------------------
# Run the Telecom Agent
# Each invocation is traced in LangSmith automatically.
# ---------------------------------------------------------------

def run_telecom_agent(query):
    """Run a query through the telecom agent and print the response."""
    print(f"Query: {query}")
    print("-" * 60)
    result = telecom_agent.invoke(
        {"messages": [HumanMessage(content=query)]}
    )
    # Print the final AI response
    final_message = result["messages"][-1]
    print(f"Response: {final_message.content}")
    print("=" * 60)
    print()
    return result


# Test the agent with different telecom queries
run_telecom_agent("What is the network status in the West region?")
run_telecom_agent("Look up customer CUST002 and tell me their plan details.")
run_telecom_agent("Calculate the bill for customer CUST001 with extra charges of 150 rupees.")

---
## Section 3: Telecom Fault Diagnosis Agent

A more advanced agent that diagnoses network faults. This demonstrates
an agent with domain-specific tools and structured reasoning.

In [ ]:
# ---------------------------------------------------------------
# Fault Diagnosis Tools
# These tools simulate telecom network diagnostics.
# ---------------------------------------------------------------

@tool
def run_ping_test(target_ip: str) -> str:
    """Run a ping test to check connectivity to a target IP address.
    Returns latency and packet loss statistics.
    
    Args:
        target_ip: The IP address to ping (e.g., '10.0.1.1')
    """
    # Simulated ping results
    results = {
        "10.0.1.1": {"latency_ms": 5, "packet_loss": "0%", "status": "reachable"},
        "10.0.2.1": {"latency_ms": 250, "packet_loss": "30%", "status": "degraded"},
        "10.0.3.1": {"latency_ms": 0, "packet_loss": "100%", "status": "unreachable"},
        "192.168.1.1": {"latency_ms": 2, "packet_loss": "0%", "status": "reachable"},
    }
    ip = target_ip.strip()
    if ip in results:
        return json.dumps({"target": ip, **results[ip]})
    # Default: simulate a random response
    return json.dumps({
        "target": ip,
        "latency_ms": random.randint(1, 500),
        "packet_loss": f"{random.randint(0, 100)}%",
        "status": "unknown",
    })


@tool
def check_tower_health(tower_id: str) -> str:
    """Check the health status of a cell tower.
    Returns power, temperature, and connection metrics.
    
    Args:
        tower_id: The cell tower identifier (e.g., 'TWR-001')
    """
    towers = {
        "TWR-001": {
            "location": "Mumbai Central",
            "power_status": "Normal",
            "temperature_c": 42,
            "active_connections": 1250,
            "max_connections": 2000,
            "status": "Healthy",
        },
        "TWR-002": {
            "location": "Delhi South",
            "power_status": "Battery Backup",
            "temperature_c": 68,
            "active_connections": 1980,
            "max_connections": 2000,
            "status": "Overloaded",
        },
        "TWR-003": {
            "location": "Bangalore East",
            "power_status": "Offline",
            "temperature_c": 0,
            "active_connections": 0,
            "max_connections": 2000,
            "status": "Down",
        },
    }
    tid = tower_id.upper().strip()
    if tid in towers:
        return json.dumps({"tower_id": tid, **towers[tid]})
    return json.dumps({"error": f"Tower not found: {tower_id}"})


@tool
def create_incident_ticket(
    severity: str,
    description: str,
    affected_region: str
) -> str:
    """Create an incident ticket in the network operations system.
    
    Args:
        severity: Ticket severity level ('low', 'medium', 'high', 'critical')
        description: Description of the incident
        affected_region: The region affected by the incident
    """
    ticket_id = f"INC-{random.randint(10000, 99999)}"
    return json.dumps({
        "ticket_id": ticket_id,
        "severity": severity,
        "description": description,
        "affected_region": affected_region,
        "status": "Open",
        "assigned_to": "Network Operations Center",
    })


diagnosis_tools = [run_ping_test, check_tower_health, create_incident_ticket]
print("Fault diagnosis tools defined: run_ping_test, check_tower_health, create_incident_ticket")

In [ ]:
# ---------------------------------------------------------------
# Build the Fault Diagnosis Agent
# ---------------------------------------------------------------

diag_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
diag_llm_with_tools = diag_llm.bind_tools(diagnosis_tools)
diag_tools_by_name = {t.name: t for t in diagnosis_tools}


def diag_call_model(state: AgentState):
    """Call the LLM for fault diagnosis."""
    system_msg = SystemMessage(
        content=(
            "You are a telecom network fault diagnosis specialist. "
            "When a user reports an issue, systematically diagnose it by: "
            "1. Running connectivity tests (ping) to identify the problem scope "
            "2. Checking tower health for the affected area "
            "3. If a fault is confirmed, create an incident ticket with appropriate severity "
            "Be thorough and explain your diagnosis step by step."
        )
    )
    response = diag_llm_with_tools.invoke([system_msg] + state["messages"])
    return {"messages": [response]}


def diag_call_tools(state: AgentState):
    """Execute diagnosis tool calls."""
    last_message = state["messages"][-1]
    results = []
    for tool_call in last_message.tool_calls:
        tool_fn = diag_tools_by_name[tool_call["name"]]
        result = tool_fn.invoke(tool_call["args"])
        results.append(
            ToolMessage(content=str(result), tool_call_id=tool_call["id"])
        )
    return {"messages": results}


def diag_should_continue(state: AgentState) -> Literal["diag_call_tools", "__end__"]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "diag_call_tools"
    return END


# Build and compile the diagnosis agent
diag_builder = StateGraph(AgentState)
diag_builder.add_node("diag_call_model", diag_call_model)
diag_builder.add_node("diag_call_tools", diag_call_tools)
diag_builder.add_edge(START, "diag_call_model")
diag_builder.add_conditional_edges(
    "diag_call_model", diag_should_continue, ["diag_call_tools", END]
)
diag_builder.add_edge("diag_call_tools", "diag_call_model")

diagnosis_agent = diag_builder.compile()
print("Fault diagnosis agent compiled successfully.")

In [ ]:
# ---------------------------------------------------------------
# Run Fault Diagnosis Scenarios
# ---------------------------------------------------------------

def run_diagnosis(issue_description):
    """Run a fault diagnosis through the agent."""
    print(f"Reported Issue: {issue_description}")
    print("-" * 60)
    result = diagnosis_agent.invoke(
        {"messages": [HumanMessage(content=issue_description)]}
    )
    final_message = result["messages"][-1]
    print(f"Diagnosis:\n{final_message.content}")
    print("=" * 60)
    print()
    return result


# Scenario 1: Tower outage
run_diagnosis(
    "Users in Bangalore East are reporting complete loss of service. "
    "Please diagnose the issue. Check tower TWR-003 and run a ping test to 10.0.3.1."
)

# Scenario 2: Degraded performance
run_diagnosis(
    "Customers in Delhi South are complaining about slow data speeds and dropped calls. "
    "Please investigate tower TWR-002 and test connectivity to 10.0.2.1."
)

---
## Section 4: Orchestrator Agent Pattern (Agentic AI)

This demonstrates the Agentic AI pattern where an orchestrator agent receives
user queries and routes them to specialized sub-agents:
- Network Operations Agent (checks network, towers)
- Customer Service Agent (looks up customers, calculates bills)
- Fault Diagnosis Agent (diagnoses issues, creates tickets)

This is a real-world pattern used in telecom NOC (Network Operations Center) systems.

In [ ]:
# ---------------------------------------------------------------
# Orchestrator Agent - Routes to Specialized Sub-Agents
# This demonstrates the Agentic AI pattern with multiple
# cooperating agents managed by an orchestrator.
# ---------------------------------------------------------------

from langgraph.graph import StateGraph, START, END


class OrchestratorState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    route: str  # Which sub-agent to use


# The orchestrator LLM decides which department handles the query
orchestrator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def classify_query(state: OrchestratorState):
    """Classify the user query and decide which sub-agent should handle it."""
    system_msg = SystemMessage(
        content=(
            "You are a telecom operations router. Classify the user query into one of these categories. "
            "Reply with ONLY the category name, nothing else.\n\n"
            "Categories:\n"
            "- NETWORK_OPS: Questions about network status, tower health, regional performance\n"
            "- CUSTOMER_SERVICE: Questions about customer accounts, plans, bills, usage\n"
            "- FAULT_DIAGNOSIS: Reports of outages, slow speeds, dropped calls, service disruptions\n"
            "- GENERAL: General questions that do not fit the above categories"
        )
    )
    response = orchestrator_llm.invoke([system_msg] + state["messages"])
    route = response.content.strip().upper()
    
    # Validate the route
    valid_routes = ["NETWORK_OPS", "CUSTOMER_SERVICE", "FAULT_DIAGNOSIS", "GENERAL"]
    if route not in valid_routes:
        route = "GENERAL"
    
    return {"route": route, "messages": []}


def route_to_agent(state: OrchestratorState) -> str:
    """Route to the appropriate sub-agent based on classification."""
    route = state.get("route", "GENERAL")
    if route == "NETWORK_OPS":
        return "network_ops_agent"
    elif route == "CUSTOMER_SERVICE":
        return "customer_service_agent"
    elif route == "FAULT_DIAGNOSIS":
        return "fault_diagnosis_agent"
    else:
        return "general_agent"


# Sub-agent nodes (each uses its own specialized tools and system prompt)

def network_ops_agent(state: OrchestratorState):
    """Handles network operations queries."""
    net_tools = [check_network_status, check_tower_health]
    net_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(net_tools)
    net_tools_map = {t.name: t for t in net_tools}
    
    system_msg = SystemMessage(
        content="You are a network operations specialist. Check network and tower status to answer queries."
    )
    msgs = [system_msg] + state["messages"]
    
    # Simple tool-calling loop (max 3 iterations)
    for _ in range(3):
        response = net_llm.invoke(msgs)
        msgs.append(response)
        if not response.tool_calls:
            break
        for tc in response.tool_calls:
            result = net_tools_map[tc["name"]].invoke(tc["args"])
            msgs.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
    
    return {"messages": [response]}


def customer_service_agent(state: OrchestratorState):
    """Handles customer service queries."""
    cust_tools = [lookup_customer, calculate_bill]
    cust_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(cust_tools)
    cust_tools_map = {t.name: t for t in cust_tools}
    
    system_msg = SystemMessage(
        content="You are a customer service agent. Help with account lookups, plan details, and billing."
    )
    msgs = [system_msg] + state["messages"]
    
    for _ in range(3):
        response = cust_llm.invoke(msgs)
        msgs.append(response)
        if not response.tool_calls:
            break
        for tc in response.tool_calls:
            result = cust_tools_map[tc["name"]].invoke(tc["args"])
            msgs.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
    
    return {"messages": [response]}


def fault_diagnosis_agent_node(state: OrchestratorState):
    """Handles fault diagnosis queries using the previously built diagnosis agent."""
    result = diagnosis_agent.invoke({"messages": state["messages"]})
    final_msg = result["messages"][-1]
    return {"messages": [final_msg]}


def general_agent(state: OrchestratorState):
    """Handles general queries that do not require specialized tools."""
    gen_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    system_msg = SystemMessage(
        content=(
            "You are a telecom operations assistant. Answer general questions about "
            "telecom operations, technology, and industry practices."
        )
    )
    response = gen_llm.invoke([system_msg] + state["messages"])
    return {"messages": [response]}


# Build the orchestrator graph
orch_builder = StateGraph(OrchestratorState)

# Add nodes
orch_builder.add_node("classify_query", classify_query)
orch_builder.add_node("network_ops_agent", network_ops_agent)
orch_builder.add_node("customer_service_agent", customer_service_agent)
orch_builder.add_node("fault_diagnosis_agent", fault_diagnosis_agent_node)
orch_builder.add_node("general_agent", general_agent)

# Add edges
orch_builder.add_edge(START, "classify_query")
orch_builder.add_conditional_edges(
    "classify_query",
    route_to_agent,
    ["network_ops_agent", "customer_service_agent", "fault_diagnosis_agent", "general_agent"],
)
orch_builder.add_edge("network_ops_agent", END)
orch_builder.add_edge("customer_service_agent", END)
orch_builder.add_edge("fault_diagnosis_agent", END)
orch_builder.add_edge("general_agent", END)

# Compile
orchestrator = orch_builder.compile()
print("Orchestrator agent compiled with 4 sub-agents:")
print("  - Network Operations")
print("  - Customer Service")
print("  - Fault Diagnosis")
print("  - General")

In [ ]:
# ---------------------------------------------------------------
# Visualize the Orchestrator Graph
# ---------------------------------------------------------------

try:
    from IPython.display import Image, display
    display(Image(orchestrator.get_graph(xray=True).draw_mermaid_png()))
except Exception as e:
    print("Orchestrator graph (Mermaid format):")
    print(orchestrator.get_graph(xray=True).draw_mermaid())

In [ ]:
# ---------------------------------------------------------------
# Run the Orchestrator with Different Query Types
# The orchestrator automatically routes each query to the right agent.
# All interactions are traced in LangSmith.
# ---------------------------------------------------------------

def run_orchestrator(query):
    """Run a query through the orchestrator and print the routed response."""
    print(f"Query: {query}")
    print("-" * 60)
    result = orchestrator.invoke(
        {"messages": [HumanMessage(content=query)]}
    )
    # Print routing decision
    print(f"Routed to: {result.get('route', 'unknown')}")
    final_message = result["messages"][-1]
    print(f"Response: {final_message.content}")
    print("=" * 60)
    print()
    return result


# Network operations query
run_orchestrator("What is the current network status in the South region?")

# Customer service query
run_orchestrator("Look up customer CUST003 and calculate their total bill.")

# Fault diagnosis query
run_orchestrator(
    "Multiple users in Bangalore East are reporting no service. "
    "Tower TWR-003 might be down. Please investigate."
)

# General query
run_orchestrator("What is the difference between 4G and 5G technology?")

---
## Section 5: Input Guardrails

Guardrails validate user input before it reaches the agent.
This prevents the agent from processing harmful or off-topic requests.

In [ ]:
# ---------------------------------------------------------------
# Guardrail Implementation
# Blocks harmful or irrelevant queries before agent processing.
# ---------------------------------------------------------------

BLOCKED_KEYWORDS = [
    "hack", "exploit", "malware", "virus", "attack",
    "steal", "password", "credential", "phishing",
]


def input_guardrail(user_input: str) -> dict:
    """Check user input against blocked keywords and content policies.
    Returns a dict with 'allowed' (bool) and 'reason' (str if blocked)."""
    input_lower = user_input.lower()
    for keyword in BLOCKED_KEYWORDS:
        if keyword in input_lower:
            return {
                "allowed": False,
                "reason": f"Query blocked: contains prohibited keyword '{keyword}'.",
            }
    if len(user_input.strip()) < 3:
        return {
            "allowed": False,
            "reason": "Query too short. Please provide more details.",
        }
    return {"allowed": True, "reason": ""}


def run_with_guardrails(query):
    """Run a query through guardrails first, then the orchestrator."""
    print(f"Query: {query}")
    
    # Check guardrail
    check = input_guardrail(query)
    if not check["allowed"]:
        print(f"BLOCKED: {check['reason']}")
        print("=" * 60)
        print()
        return None
    
    print("Guardrail: PASSED")
    return run_orchestrator(query)


# Test guardrails
run_with_guardrails("How do I hack into the network system?")
run_with_guardrails("Check the network status in the North region.")

---
## Section 6: LangSmith Tracing for Agents

All agent calls above are automatically traced by LangSmith.
Here we show additional tracing features specific to agent workflows.

In [ ]:
# ---------------------------------------------------------------
# LangSmith Client Verification
# ---------------------------------------------------------------

from langsmith import Client as LangSmithClient

ls_client = LangSmithClient()

try:
    projects = list(ls_client.list_projects(limit=5))
    print("LangSmith connection verified.")
    print(f"Active project: {os.environ.get('LANGSMITH_PROJECT', 'default')}")
    print(f"Total projects: {len(projects)}")
    print()
    print("All agent invocations above have been traced.")
    print("View traces at: https://smith.langchain.com")
    print()
    print("In LangSmith you can see:")
    print("  - The orchestrator routing decision")
    print("  - Which sub-agent handled each query")
    print("  - Every tool call and its arguments/results")
    print("  - Token usage and latency for each LLM call")
except Exception as e:
    print(f"LangSmith connection issue: {e}")

In [ ]:
# ---------------------------------------------------------------
# Explicit Tracing with @traceable for Custom Agent Logic
# ---------------------------------------------------------------

from langsmith import traceable


@traceable(name="TelecomAgentPipeline", run_type="chain")
def traced_telecom_pipeline(query):
    """Run the full telecom pipeline with explicit tracing.
    This creates a parent span in LangSmith containing:
    - Guardrail check
    - Query classification
    - Sub-agent execution
    - Final response
    """
    # Step 1: Guardrail
    guard_check = input_guardrail(query)
    if not guard_check["allowed"]:
        return {"query": query, "blocked": True, "reason": guard_check["reason"]}
    
    # Step 2: Run through orchestrator
    result = orchestrator.invoke({"messages": [HumanMessage(content=query)]})
    final_msg = result["messages"][-1].content
    
    return {
        "query": query,
        "blocked": False,
        "route": result.get("route", "unknown"),
        "response": final_msg,
    }


# Run traced pipeline
pipeline_result = traced_telecom_pipeline(
    "What is the network status in the East region?"
)

print("Traced pipeline result:")
for key, value in pipeline_result.items():
    display_value = str(value)[:200] if len(str(value)) > 200 else str(value)
    print(f"  {key}: {display_value}")

print()
print("This trace appears in LangSmith as 'TelecomAgentPipeline'")
print("with all sub-steps visible as child spans.")

In [ ]:
# ---------------------------------------------------------------
# Adding Tags and Metadata to Traces
# Useful for filtering and grouping traces in LangSmith.
# ---------------------------------------------------------------

import langsmith as ls

# Run with custom tags and metadata for easier filtering in LangSmith
with ls.tracing_context(
    project_name="langgraph-agents-telecom",
    tags=["production-test", "orchestrator"],
    metadata={"environment": "notebook", "version": "1.0"},
    enabled=True,
):
    result = orchestrator.invoke(
        {"messages": [HumanMessage(content="Check tower TWR-001 health status.")]}
    )
    print(f"Response: {result['messages'][-1].content}")

print()
print("This trace has tags ['production-test', 'orchestrator'] in LangSmith.")
print("Filter by these tags in the LangSmith UI to find specific trace groups.")

---
## Summary

### Concepts Covered

**Agent vs Agentic AI**: An agent is a single LLM-tool loop for a specific task.
Agentic AI is a design pattern combining multiple agents with planning, memory,
and orchestration for complex workflows.

### Agents Built

**Network Status Agent**: Checks regional network health and tower metrics.

**Customer Service Agent**: Looks up customer accounts and calculates bills.

**Fault Diagnosis Agent**: Systematically diagnoses network faults using ping tests,
tower health checks, and creates incident tickets.

**Orchestrator Agent**: Routes incoming queries to the appropriate specialized agent,
demonstrating the agentic AI pattern used in telecom NOC systems.

### LangSmith Integration

All agent invocations are automatically traced. Custom @traceable decorators,
tags, and metadata enable fine-grained observability and debugging in LangSmith.